In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/datasets/kamrun71/feature-3/feature_combiner_3.pkl
/kaggle/input/datasets/kamrun71/retrained-models/retrained_random_forest.pkl
/kaggle/input/datasets/kamrun71/retrained-models/retrained_xgboost.pkl
/kaggle/input/datasets/kamrun71/retrained-models/retrained_logistic_regression.pkl
/kaggle/input/datasets/kamrun71/retrained-models/final_feature_indices.pkl
/kaggle/input/datasets/kamrun71/retrained-models/retrained_linearsvc.pkl
/kaggle/input/datasets/kamrun71/final-models/linearsvc.joblib
/kaggle/input/datasets/kamrun71/final-models/xgboost.joblib
/kaggle/input/datasets/kamrun71/final-models/random_forest.joblib
/kaggle/input/datasets/kamrun71/final-models/logistic_regression.joblib
/kaggle/input/datasets/kamrun71/processed-data/y_train.npy
/kaggle/input/datasets/kamrun71/processed-data/y_test.npy
/kaggle/input/datasets/kamrun71/processed-data/y_val.npy
/kaggle/input/datasets/kamrun71/processed-data/X_test_vec.npy
/kaggle/input/datasets/kamrun71/processed-data/X_val_vec.npy

In [2]:
"""
=============================================================
  COMPLETE CV STATISTICAL TEST — ALL MODELS
  Base | Retrained | Ensemble
  Metrics: Accuracy, ROC-AUC, F1-Macro
  CV: 25-Fold Stratified

  IMPORTANT: This script saves fold-level arrays for ALL
  models so they can be used for:
  - ANOVA / Friedman tests
  - Paired t-test / Wilcoxon between phases
  - Any future statistical comparisons
=============================================================
"""

import numpy as np
import joblib
import warnings
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from scipy.stats import wilcoxon, f_oneway, friedmanchisquare
from scipy.stats import ttest_rel
from joblib import Parallel, delayed
import time

warnings.filterwarnings("ignore")

# =============================================================
# 1. CUSTOM CLASSES
# =============================================================

class PreFittedVoting:
    def __init__(self, named_models):
        self.named_models = named_models

    def predict_proba(self, X):
        probas = []
        for name, model in self.named_models:
            if "LinearSVC" in name:
                d = model.decision_function(X)
                p = 1 / (1 + np.exp(-d))
                probas.append(np.column_stack([1 - p, p]))
            else:
                probas.append(model.predict_proba(X))
        return np.mean(probas, axis=0)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    def decision_function(self, X):
        proba = self.predict_proba(X)[:, 1]
        return np.log(proba / (1 - proba + 1e-9))


class CrossFeatureVoting:
    def __init__(self, full_models, opt_models):
        self.full_models = full_models
        self.opt_models  = opt_models

    def predict_proba(self, X_full, X_opt):
        probas = []
        for name, model in self.full_models:
            if "LinearSVC" in name:
                d = model.decision_function(X_full)
                p = 1 / (1 + np.exp(-d))
                probas.append(np.column_stack([1 - p, p]))
            else:
                probas.append(model.predict_proba(X_full))
        for name, model in self.opt_models:
            if "LinearSVC" in name:
                d = model.decision_function(X_opt)
                p = 1 / (1 + np.exp(-d))
                probas.append(np.column_stack([1 - p, p]))
            else:
                probas.append(model.predict_proba(X_opt))
        return np.mean(probas, axis=0)

    def predict(self, X_full, X_opt):
        return (self.predict_proba(X_full, X_opt)[:, 1] >= 0.5).astype(int)


# =============================================================
# 2. LOAD DATA
# =============================================================

def load_npy(path):
    arr = np.load(path, allow_pickle=True)
    if arr.ndim == 0:
        arr = arr.item()
    return arr

print("Loading saved splits...")
X_train = load_npy("/kaggle/input/datasets/kamrun71/processed-data/X_train_vec.npy")
X_test  = load_npy("/kaggle/input/datasets/kamrun71/processed-data/X_test_vec.npy")
X_val   = load_npy("/kaggle/input/datasets/kamrun71/processed-data/X_val_vec.npy")
y_train = np.array(load_npy("/kaggle/input/datasets/kamrun71/processed-data/y_train.npy"))
y_test  = np.array(load_npy("/kaggle/input/datasets/kamrun71/processed-data/y_test.npy"))
y_val   = np.array(load_npy("/kaggle/input/datasets/kamrun71/processed-data/y_val.npy"))

if sp.issparse(X_train):
    X_cv = sp.vstack([X_train, X_val]).tocsr()
else:
    X_cv = np.concatenate([X_train, X_val], axis=0)

y_cv = np.concatenate([y_train, y_val], axis=0)
print(f"  CV shape : {X_cv.shape} | Sparse: {sp.issparse(X_cv)}")

feature_indices = joblib.load(
    "/kaggle/input/datasets/kamrun71/retrained-models/final_feature_indices.pkl"
)
if sp.issparse(X_cv):
    X_cv_opt   = X_cv[:, feature_indices].tocsr()
    X_test_opt = X_test[:, feature_indices].tocsr()
else:
    X_cv_opt   = X_cv[:, feature_indices]
    X_test_opt = X_test[:, feature_indices]

print("  Converting to dense for ensemble fold loops...")
X_cv_dense     = X_cv.toarray()     if sp.issparse(X_cv)     else X_cv
X_cv_opt_dense = X_cv_opt.toarray() if sp.issparse(X_cv_opt) else X_cv_opt
print(f"  Dense shapes: full={X_cv_dense.shape}, opt={X_cv_opt_dense.shape}")

# =============================================================
# 3. LOAD ALL MODELS
# =============================================================
print("\nLoading models...")

base_lr  = joblib.load("/kaggle/input/datasets/kamrun71/final-models/logistic_regression.joblib")
base_rf  = joblib.load("/kaggle/input/datasets/kamrun71/final-models/random_forest.joblib")
base_svc = joblib.load("/kaggle/input/datasets/kamrun71/final-models/linearsvc.joblib")
base_xgb = joblib.load("/kaggle/input/datasets/kamrun71/final-models/xgboost.joblib")

ret_lr   = joblib.load("/kaggle/input/datasets/kamrun71/retrained-models/retrained_logistic_regression.pkl")
ret_rf   = joblib.load("/kaggle/input/datasets/kamrun71/retrained-models/retrained_random_forest.pkl")
ret_svc  = joblib.load("/kaggle/input/datasets/kamrun71/retrained-models/retrained_linearsvc.pkl")
ret_xgb  = joblib.load("/kaggle/input/datasets/kamrun71/retrained-models/retrained_xgboost.pkl")

ens_full         = joblib.load("/kaggle/input/datasets/kamrun71/ensemble-models/ensemble_full_features.pkl")
ens_opt          = joblib.load("/kaggle/input/datasets/kamrun71/ensemble-models/ensemble_opt_features.pkl")
cross_lr_svc_xgb = joblib.load("/kaggle/input/datasets/kamrun71/ensemble-models/cross_ensemble_lr_full_svc_xgb_opt.pkl")
cross_lr_xgb     = joblib.load("/kaggle/input/datasets/kamrun71/ensemble-models/cross_ensemble_lr_xgb.pkl")
cross_all        = joblib.load("/kaggle/input/datasets/kamrun71/ensemble-models/cross_ensemble_lr_xgb_full_svc_xgb_opt.pkl")

print("  All models loaded.")

# =============================================================
# 4. CV SETUP
# =============================================================
CV_FOLDS = 25
skf      = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
SCORING  = ["accuracy", "roc_auc", "f1_macro"]

# =============================================================
# 5. CV RUNNERS
# =============================================================

def wrap_for_cv(model):
    if isinstance(model, LinearSVC):
        return CalibratedClassifierCV(model, cv=3)
    return model


def run_cv_standard(model, X, y, model_name, feature_set_label):
    """Standard sklearn cross_validate — returns fold arrays."""
    m = wrap_for_cv(model)
    start = time.time()
    cv_res = cross_validate(
        m, X, y,
        cv=skf,
        scoring=SCORING,
        return_train_score=True,
        n_jobs=-1,
        error_score="raise"
    )
    elapsed = time.time() - start

    # Extract fold arrays
    acc_val = cv_res["test_accuracy"]
    auc_val = cv_res["test_roc_auc"]
    f1_val  = cv_res["test_f1_macro"]
    acc_tr  = cv_res["train_accuracy"]
    auc_tr  = cv_res["train_roc_auc"]
    f1_tr   = cv_res["train_f1_macro"]

    row = _build_row_from_arrays(
        acc_tr, auc_tr, f1_tr,
        acc_val, auc_val, f1_val,
        model_name, feature_set_label, "Standard CV", elapsed
    )
    return row


def _single_fold_single(model, X, y, train_idx, val_idx):
    X_tr, X_vl = X[train_idx], X[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    tr_pred  = model.predict(X_tr)
    tr_proba = model.predict_proba(X_tr)[:, 1]
    vl_pred  = model.predict(X_vl)
    vl_proba = model.predict_proba(X_vl)[:, 1]
    return (
        accuracy_score(y_tr, tr_pred),
        roc_auc_score(y_tr,  tr_proba),
        f1_score(y_tr, tr_pred, average="macro", zero_division=0),
        accuracy_score(y_vl, vl_pred),
        roc_auc_score(y_vl,  vl_proba),
        f1_score(y_vl, vl_pred, average="macro", zero_division=0),
    )


def _single_fold_cross(model, X_full, X_opt, y, train_idx, val_idx):
    Xf_tr, Xf_vl = X_full[train_idx], X_full[val_idx]
    Xo_tr, Xo_vl = X_opt[train_idx],  X_opt[val_idx]
    y_tr,  y_vl  = y[train_idx],      y[val_idx]
    tr_pred  = model.predict(Xf_tr, Xo_tr)
    tr_proba = model.predict_proba(Xf_tr, Xo_tr)[:, 1]
    vl_pred  = model.predict(Xf_vl, Xo_vl)
    vl_proba = model.predict_proba(Xf_vl, Xo_vl)[:, 1]
    return (
        accuracy_score(y_tr, tr_pred),
        roc_auc_score(y_tr,  tr_proba),
        f1_score(y_tr, tr_pred, average="macro", zero_division=0),
        accuracy_score(y_vl, vl_pred),
        roc_auc_score(y_vl,  vl_proba),
        f1_score(y_vl, vl_pred, average="macro", zero_division=0),
    )


def run_cv_prefitted(model, X, y, model_name, feature_set_label):
    folds  = list(skf.split(X, y))
    start  = time.time()
    res    = Parallel(n_jobs=-1)(
        delayed(_single_fold_single)(model, X, y, tr, vl) for tr, vl in folds
    )
    elapsed = time.time() - start
    return _build_row_from_tuples(res, model_name, feature_set_label, "PreFitted CV", elapsed)


def run_cv_cross_feature(model, X_full, X_opt, y, model_name):
    folds  = list(skf.split(X_full, y))
    start  = time.time()
    res    = Parallel(n_jobs=-1)(
        delayed(_single_fold_cross)(model, X_full, X_opt, y, tr, vl) for tr, vl in folds
    )
    elapsed = time.time() - start
    return _build_row_from_tuples(res, model_name, "Mixed (Full + Opt)", "Cross-Feature CV", elapsed)


def _build_row_from_tuples(fold_results, model_name, feature_set_label, cv_type, elapsed):
    acc_tr  = np.array([r[0] for r in fold_results])
    auc_tr  = np.array([r[1] for r in fold_results])
    f1_tr   = np.array([r[2] for r in fold_results])
    acc_val = np.array([r[3] for r in fold_results])
    auc_val = np.array([r[4] for r in fold_results])
    f1_val  = np.array([r[5] for r in fold_results])
    return _build_row_from_arrays(
        acc_tr, auc_tr, f1_tr,
        acc_val, auc_val, f1_val,
        model_name, feature_set_label, cv_type, elapsed
    )


def _build_row_from_arrays(acc_tr, auc_tr, f1_tr,
                            acc_val, auc_val, f1_val,
                            model_name, feature_set_label, cv_type, elapsed):
    gap_acc = acc_tr.mean() - acc_val.mean()
    gap_auc = auc_tr.mean() - auc_val.mean()
    gap_f1  = f1_tr.mean()  - f1_val.mean()

    row = {
        "Model"              : model_name,
        "Feature Set"        : feature_set_label,
        "CV Time (s)"        : round(elapsed, 1),
        "Type"               : cv_type,

        # Accuracy
        "Train ACCURACY Mean": round(acc_tr.mean(),  4),
        "Train ACCURACY Std" : round(acc_tr.std(),   4),
        "Val ACCURACY Mean"  : round(acc_val.mean(), 4),
        "Val ACCURACY Std"   : round(acc_val.std(),  4),
        "Gap ACCURACY"       : round(gap_acc, 4),
        "Overfit? (ACCURACY)": "⚠ YES" if gap_acc > 0.05 else "⚠ UNDERFIT" if gap_acc < -0.01 else " OK",

        # AUC
        "Train ROC-AUC Mean" : round(auc_tr.mean(),  4),
        "Train ROC-AUC Std"  : round(auc_tr.std(),   4),
        "Val ROC-AUC Mean"   : round(auc_val.mean(), 4),
        "Val ROC-AUC Std"    : round(auc_val.std(),  4),
        "Gap ROC-AUC"        : round(gap_auc, 4),
        "Overfit? (ROC-AUC)" : "⚠ YES" if gap_auc > 0.05 else "⚠ UNDERFIT" if gap_auc < -0.01 else " OK",

        # F1
        "Train F1-MACRO Mean": round(f1_tr.mean(),  4),
        "Train F1-MACRO Std" : round(f1_tr.std(),   4),
        "Val F1-MACRO Mean"  : round(f1_val.mean(), 4),
        "Val F1-MACRO Std"   : round(f1_val.std(),  4),
        "Gap F1-MACRO"       : round(gap_f1, 4),
        "Overfit? (F1-MACRO)": "⚠ YES" if gap_f1 > 0.05 else "⚠ UNDERFIT" if gap_f1 < -0.01 else " OK",

        # Raw fold arrays — CRITICAL for ANOVA / Friedman / t-test / Wilcoxon
        "_acc_val_folds"     : acc_val,
        "_auc_val_folds"     : auc_val,
        "_f1_val_folds"      : f1_val,
        "_acc_tr_folds"      : acc_tr,
    }

    print(f"   {model_name:<45} | Acc: {row['Val ACCURACY Mean']:.4f} ± "
          f"{row['Val ACCURACY Std']:.4f} | Gap: {row['Gap ACCURACY']:.4f} | {elapsed:.1f}s")
    return row


# =============================================================
# 6. RUN ALL GROUPS
# =============================================================
all_results = []

print("\n" + "="*60)
print("  GROUP 1: BASE MODELS (Full Feature Set)")
print("="*60)
all_results.append(run_cv_standard(base_lr,  X_cv, y_cv, "LR (Base)",        "Full"))
all_results.append(run_cv_standard(base_rf,  X_cv, y_cv, "RF (Base)",        "Full"))
all_results.append(run_cv_standard(base_svc, X_cv, y_cv, "LinearSVC (Base)", "Full"))
all_results.append(run_cv_standard(base_xgb, X_cv, y_cv, "XGBoost (Base)",   "Full"))

print("\n" + "="*60)
print("  GROUP 2: RETRAINED MODELS (Optimized Feature Set)")
print("="*60)
all_results.append(run_cv_standard(ret_lr,  X_cv_opt, y_cv, "LR (Retrained)",        "Optimized"))
all_results.append(run_cv_standard(ret_rf,  X_cv_opt, y_cv, "RF (Retrained)",        "Optimized"))
all_results.append(run_cv_standard(ret_svc, X_cv_opt, y_cv, "LinearSVC (Retrained)", "Optimized"))
all_results.append(run_cv_standard(ret_xgb, X_cv_opt, y_cv, "XGBoost (Retrained)",   "Optimized"))

print("\n" + "="*60)
print("  GROUP 3: ENSEMBLE MODELS (PreFittedVoting)")
print("="*60)
all_results.append(run_cv_prefitted(ens_full, X_cv_dense,     y_cv, "Ensemble (Full)", "Full"))
all_results.append(run_cv_prefitted(ens_opt,  X_cv_opt_dense, y_cv, "Ensemble (Opt)",  "Optimized"))

print("\n" + "="*60)
print("  GROUP 4: CROSS-FEATURE ENSEMBLES")
print("="*60)
all_results.append(run_cv_cross_feature(cross_lr_xgb,     X_cv_dense, X_cv_opt_dense, y_cv, "Cross Ensemble LR+XGB"))
all_results.append(run_cv_cross_feature(cross_lr_svc_xgb, X_cv_dense, X_cv_opt_dense, y_cv, "Cross Ensemble LR+SVC+XGB"))
all_results.append(run_cv_cross_feature(cross_all,        X_cv_dense, X_cv_opt_dense, y_cv, "Cross Ensemble All"))

# =============================================================
# 7. EXTRACT FOLD ARRAYS INTO SEPARATE DICT
# =============================================================
fold_arrays = {}
for r in all_results:
    name = r["Model"]
    fold_arrays[name] = {
        "acc" : r.pop("_acc_val_folds"),
        "auc" : r.pop("_auc_val_folds"),
        "f1"  : r.pop("_f1_val_folds"),
        "acc_train": r.pop("_acc_tr_folds"),
    }

# Save fold arrays for later use (P1/P2 comparison etc.)
joblib.dump(fold_arrays, "p3_fold_arrays.pkl")
print("\n  Saved fold arrays → p3_fold_arrays.pkl")

df = pd.DataFrame(all_results)

# =============================================================
# 8. WILCOXON — best ensemble vs all others
# =============================================================
print("\n" + "="*60)
print("  WILCOXON SIGNED-RANK TEST (All models vs Best Ensemble)")
print("="*60)

best_name  = max(fold_arrays, key=lambda k: fold_arrays[k]["acc"].mean())
best_folds = fold_arrays[best_name]["acc"]
print(f"\n  Reference: {best_name} ({best_folds.mean():.4f})\n")

wilcoxon_rows = []
for name, arrays in fold_arrays.items():
    folds = arrays["acc"]
    if name == best_name:
        wilcoxon_rows.append({"Model": name, "Val Acc Mean": round(folds.mean(), 4),
                               "p-value": "—", "Significant": "— (reference)"})
        continue
    diff = best_folds - folds
    if np.all(diff == 0):
        p_val = 1.0
    else:
        _, p_val = wilcoxon(best_folds, folds, alternative="greater")
    sig = " YES" if p_val < 0.05 else "✗ NO"
    wilcoxon_rows.append({"Model": name, "Val Acc Mean": round(folds.mean(), 4),
                           "p-value": round(p_val, 4), "Significant": sig})
    print(f"  {name:<45} p={p_val:.4f}  {sig}")

df_wilcoxon = pd.DataFrame(wilcoxon_rows)

# =============================================================
# 9. ANOVA + FRIEDMAN — per model across P3 stages
#    Compares: Base vs Retrained vs Best Ensemble (per model type)
# =============================================================
print("\n" + "="*60)
print("  ANOVA + FRIEDMAN — P3 Stage Comparison per Model")
print("  (Base → Retrained → Best Ensemble)")
print("="*60)

model_types  = ["LR", "RF", "LinearSVC", "XGBoost"]
base_names   = {"LR": "LR (Base)",        "RF": "RF (Base)",
                 "LinearSVC": "LinearSVC (Base)", "XGBoost": "XGBoost (Base)"}
ret_names    = {"LR": "LR (Retrained)",    "RF": "RF (Retrained)",
                 "LinearSVC": "LinearSVC (Retrained)", "XGBoost": "XGBoost (Retrained)"}
best_ens     = best_name   # use best ensemble for all comparisons

anova_rows = []
for mt in model_types:
    b_folds  = fold_arrays[base_names[mt]]["acc"]
    r_folds  = fold_arrays[ret_names[mt]]["acc"]
    e_folds  = fold_arrays[best_ens]["acc"]

    # One-way ANOVA
    f_stat, p_anova = f_oneway(b_folds, r_folds, e_folds)

    # Friedman (non-parametric)
    chi2, p_friedman = friedmanchisquare(b_folds, r_folds, e_folds)

    anova_rows.append({
        "Model"          : mt,
        "F-Statistic"    : round(f_stat,    2),
        "ANOVA p-value"  : f"{p_anova:.2e}",
        "Chi2-Statistic" : round(chi2,      2),
        "Friedman p-value": f"{p_friedman:.2e}",
    })
    print(f"  {mt:<15} | ANOVA F={f_stat:.2f} p={p_anova:.2e} | "
          f"Friedman χ²={chi2:.2f} p={p_friedman:.2e}")

df_anova = pd.DataFrame(anova_rows)

# =============================================================
# 10. PAIRED T-TEST + WILCOXON — P3 pairwise stage comparison
#     P3 Base vs P3 Retrained
#     P3 Retrained vs P3 Ensemble
#     P3 Base vs P3 Ensemble
# =============================================================
print("\n" + "="*60)
print("  PAIRED T-TEST + WILCOXON — P3 Pairwise Stage Comparison")
print("="*60)

comparisons = [
    ("P3 Base", "P3 Retrained"),
    ("P3 Retrained", "P3 Ensemble"),
    ("P3 Base", "P3 Ensemble"),
]

# Map stage names to fold arrays per model type
def get_stage_folds(model_type, stage):
    if stage == "P3 Base":
        return fold_arrays[base_names[model_type]]["acc"]
    elif stage == "P3 Retrained":
        return fold_arrays[ret_names[model_type]]["acc"]
    elif stage == "P3 Ensemble":
        return fold_arrays[best_ens]["acc"]

paired_rows = []
for mt in model_types:
    for (s1, s2) in comparisons:
        f1_arr = get_stage_folds(mt, s1)
        f2_arr = get_stage_folds(mt, s2)

        mean_gain = (f2_arr.mean() - f1_arr.mean()) * 100

        # Paired t-test
        t_stat, p_t = ttest_rel(f1_arr, f2_arr)

        # Wilcoxon
        diff = f2_arr - f1_arr
        if np.all(diff == 0):
            w_stat, p_w = 0.0, 1.0
        else:
            w_stat, p_w = wilcoxon(f1_arr, f2_arr)

        paired_rows.append({
            "Model"          : mt,
            "Comparison"     : f"{s1} vs {s2}",
            "t-Statistic"    : round(t_stat, 2),
            "t p-value"      : f"{p_t:.2e}",
            "W-Statistic"    : round(w_stat, 1),
            "W p-value"      : f"{p_w:.2e}",
            "Mean Gain (%)"  : round(mean_gain, 4),
        })
        print(f"  {mt:<12} {s1} vs {s2:<25} | "
              f"t={t_stat:.2f} p={p_t:.2e} | "
              f"W={w_stat:.0f} p={p_w:.2e} | "
              f"Gain={mean_gain:.4f}%")

df_paired = pd.DataFrame(paired_rows)

# =============================================================
# 11. PRINT FULL SUMMARY
# =============================================================
display_cols = [
    "Model", "Feature Set", "Type",
    "Val ACCURACY Mean", "Val ACCURACY Std", "Gap ACCURACY", "Overfit? (ACCURACY)",
    "Val ROC-AUC Mean",  "Val ROC-AUC Std",  "Gap ROC-AUC",  "Overfit? (ROC-AUC)",
    "Val F1-MACRO Mean", "Val F1-MACRO Std",  "Gap F1-MACRO",
    "CV Time (s)"
]

print("\n\n" + "="*60)
print("  FULL CV RESULTS SUMMARY")
print("="*60)
print(df[display_cols].sort_values("Val ACCURACY Mean", ascending=False).to_string(index=False))

print("\n\n" + "="*60)
print("  ANOVA + FRIEDMAN RESULTS")
print("="*60)
print(df_anova.to_string(index=False))

print("\n\n" + "="*60)
print("  PAIRED T-TEST + WILCOXON RESULTS")
print("="*60)
print(df_paired.to_string(index=False))

print("\n\n" + "="*60)
print("  WILCOXON — BEST ENSEMBLE VS ALL")
print("="*60)
print(df_wilcoxon.to_string(index=False))

# =============================================================
# 12. SAVE EVERYTHING
# =============================================================
df[display_cols].to_csv("cv_results_complete.csv", index=False)
df_wilcoxon.to_csv("wilcoxon_best_vs_all.csv", index=False)
df_anova.to_csv("anova_friedman_p3_stages.csv", index=False)
df_paired.to_csv("paired_ttest_wilcoxon_p3_stages.csv", index=False)

print("\n\n  Saved → cv_results_complete.csv")
print("  Saved → wilcoxon_best_vs_all.csv")
print("  Saved → anova_friedman_p3_stages.csv")
print("  Saved → paired_ttest_wilcoxon_p3_stages.csv")
print("  Saved → p3_fold_arrays.pkl  (use this for P1/P2 comparison later)")
print("\n  DONE.")

Loading saved splits...
  CV shape : (30037, 7782) | Sparse: True
  Converting to dense for ensemble fold loops...
  Dense shapes: full=(30037, 7782), opt=(30037, 2896)

Loading models...
  All models loaded.

  GROUP 1: BASE MODELS (Full Feature Set)
   LR (Base)                                     | Acc: 0.9696 ± 0.0056 | Gap: 0.0110 | 878.0s
   RF (Base)                                     | Acc: 0.9586 ± 0.0061 | Gap: 0.0414 | 1585.5s


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


   LinearSVC (Base)                              | Acc: 0.9682 ± 0.0043 | Gap: 0.0174 | 1112.9s
   XGBoost (Base)                                | Acc: 0.9683 ± 0.0056 | Gap: 0.0317 | 1339.0s

  GROUP 2: RETRAINED MODELS (Optimized Feature Set)
   LR (Retrained)                                | Acc: 0.9717 ± 0.0050 | Gap: 0.0067 | 274.5s
   RF (Retrained)                                | Acc: 0.9589 ± 0.0062 | Gap: 0.0411 | 1152.9s
   LinearSVC (Retrained)                         | Acc: 0.9728 ± 0.0045 | Gap: 0.0115 | 73.5s
   XGBoost (Retrained)                           | Acc: 0.9690 ± 0.0044 | Gap: 0.0310 | 580.8s

  GROUP 3: ENSEMBLE MODELS (PreFittedVoting)
   Ensemble (Full)                               | Acc: 0.9912 ± 0.0023 | Gap: -0.0000 | 90.2s
   Ensemble (Opt)                                | Acc: 0.9910 ± 0.0026 | Gap: -0.0000 | 50.0s

  GROUP 4: CROSS-FEATURE ENSEMBLES
   Cross Ensemble LR+XGB                         | Acc: 0.9906 ± 0.0028 | Gap: -0.0000 | 59.9s
   Cross